# G12 rotation (week 3): quantize and evaluate `sarvamai/sarvam-1`, time and cost

R1 runs the team flow `teams/12-quantization/scripts/quantize_eval.py` on one open-source checkpoint. R2 records the
time and the cost. Output: `runs/rotation_subramanyanaik.md` (written afterwards from the zip this notebook makes).

**Why this checkpoint:** the earlier rotation runs (#13, #16, #18 to #23) are Qwen, Granite and Llama. Sarvam-1 is the
first Indic model in the rotation (10 Indic languages and English, Llama architecture, 2.5 B weights, a 68K
SentencePiece vocabulary). So besides the stand-in score, this run measures how far each format drifts from bf16 on
**Kannada** text against **English** text (`quality/kl_eval.py`).

**How to run:** Runtime → Change runtime type → **T4 GPU** (any NVIDIA GPU works). Then Runtime → Run all.
On a free T4 it took about 35 min for command 1, about 10 min for the drift check and over an hour for
command 2 (200 items). The last cell downloads `g12_rotation_sarvam1_out.zip`.

The scripts are used as they are on the lead's branch at a pinned commit. Nothing here changes them.

In [ ]:
# 1. Settings (pinned). Change only COLAB_UNITS_PER_HOUR, from Colab's resources panel, if you are on Colab Pro.
import os, sys, time, json, hashlib, shutil, subprocess, platform, urllib.request, tarfile, datetime
from pathlib import Path

T_SESSION = time.time()
MODEL = 'sarvamai/sarvam-1'
REVISION = 'e9607337286ddf496d4a2562b194e489dcf3feea'
REPO = 'https://github.com/SreekanthZipsy/era-v5-lab'          # the lead's fork; rotation PRs go to G12-Week3
REPO_COMMIT = '26968de'                                         # G12-Week3 head on 18 Nov 2026
LLAMA_TAG = 'b11669'                                            # the build the team and the other rotation runs use
LLAMA_FILES = {  # release assets and their GitHub sha256
    'llama-b11669-bin-ubuntu-cuda-12.8-x64.tar.gz': '4703ed1a3cd6e8504affb6cb9866f501e96d634d68a002dc9a8251c9e2819b49',
    'cudart-llama-b11669-bin-ubuntu-cuda-12.8-x64.tar.gz': 'ab3eda89a82096bcba00b6448218c3c11eec2016b0f38d64884dc29736549f26',
}
FORMATS = 'bf16,Q8_0,Q6_K,Q5_K_M,Q4_K_M,Q3_K_M,Q2_K'
RUN_FULL = True        # also score all 200 stand-in items (command 2), reusing the GGUFs of command 1
RUN_KL = True          # KL drift from bf16 on English (WikiText-2) and Kannada (Sangraha verified) text
KL_CHUNKS = 64         # 64 chunks x 512 tokens: 16,384 scored tokens per text, like quality/kl.csv
SANGRAHA_REV = '8b813c3f62d37b2fa174d68c31e8b35ae2fe85e8'      # ai4bharat/sangraha, CC-BY-4.0
KN_CHARS = 400_000     # Kannada text size: whole documents, in file order, until at least this many characters
COLAB_UNITS_PER_HOUR = None   # e.g. 1.2 for a T4 on Colab Pro; None = free tier

ROOT = Path('/content'); WORK = ROOT / 'work'; OUT = ROOT / 'out'; DL = ROOT / 'dl'
REPO_DIR = ROOT / 'era-v5-lab'
for p in (WORK, OUT, DL): p.mkdir(parents=True, exist_ok=True)
STAGES = {}   # stage -> seconds, written to out/timings.json

def stage(name, t0):
    STAGES[name] = round(time.time() - t0, 1)
    (OUT / 'timings.json').write_text(json.dumps(STAGES, indent=2))
    print(f'== {name}: {STAGES[name]} s')

def sh(cmd, **kw):
    print('$', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True, text=True, **kw)

In [ ]:
# 2. The machine, and a GPU sampler (memory and utilisation every 5 s) that runs until the summary cell
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
gpu_log = open(OUT / 'gpu_log.csv', 'w')
sampler = subprocess.Popen(['nvidia-smi', '--query-gpu=timestamp,memory.used,utilization.gpu',
                            '--format=csv,noheader,nounits', '-l', '5'], stdout=gpu_log)

cpu = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), platform.machine())
ram_gb = round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1)
env = dict(gpu=gpu.stdout.strip(), cpu=cpu, cores=os.cpu_count(), ram_gb=ram_gb, python=platform.python_version(),
           kernel=platform.release(), disk_free_gb=round(shutil.disk_usage('/content').free / 1e9, 1),
           started_utc=datetime.datetime.utcnow().isoformat(timespec='seconds'))
print(json.dumps(env, indent=2))

In [ ]:
# 3. llama.cpp b11669: the prebuilt CUDA 12.8 release (no source build), checked against GitHub's sha256
t0 = time.time()
base = f'https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}/'
LLAMA_DIR = ROOT / f'llama-{LLAMA_TAG}'
for name, want in LLAMA_FILES.items():
    f = DL / name
    if not f.exists():
        urllib.request.urlretrieve(base + name, f)
    h = hashlib.sha256(f.read_bytes()).hexdigest()
    assert h == want, f'{name}: sha256 {h}, expected {want}'
    with tarfile.open(f) as t:
        t.extractall(LLAMA_DIR)
server = next(LLAMA_DIR.rglob('llama-server'))
LLAMA_BIN = server.parent
so_dirs = sorted({str(p.parent) for p in LLAMA_DIR.rglob('*.so*')})
os.environ['LD_LIBRARY_PATH'] = ':'.join([str(LLAMA_BIN), *so_dirs, os.environ.get('LD_LIBRARY_PATH', '')])
for b in LLAMA_BIN.iterdir():
    if b.is_file() and b.name.startswith('llama-'): b.chmod(0o755)
print(subprocess.run([LLAMA_BIN / 'llama-server', '--version'], capture_output=True, text=True).stderr)
devices = subprocess.run([LLAMA_BIN / 'llama-cli', '--list-devices'], capture_output=True, text=True)
print(devices.stdout + devices.stderr)
assert 'CUDA0' in devices.stdout + devices.stderr, 'llama.cpp does not see the GPU'
stage('setup_llama_cpp', t0)

In [ ]:
# 4. The team repo at the pinned G12-Week3 commit (only teams/12-quantization is checked out), and uv for the
#    script's private Python 3.12 convert venv
t0 = time.time()
if not REPO_DIR.exists():
    sh(['git', 'clone', '--filter=blob:none', '--sparse', '--no-checkout', REPO, REPO_DIR])
    sh(['git', '-C', REPO_DIR, 'sparse-checkout', 'set', 'teams/12-quantization'])
sh(['git', '-C', REPO_DIR, 'checkout', '-q', REPO_COMMIT])
REPO_SHA = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('repo at', REPO_SHA)
sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
QE = REPO_DIR / 'teams/12-quantization/scripts/quantize_eval.py'
KL = REPO_DIR / 'teams/12-quantization/quality/kl_eval.py'
env.update(repo=REPO, repo_commit=REPO_SHA, llama_tag=LLAMA_TAG, model=MODEL, revision=REVISION)
(OUT / 'environment.json').write_text(json.dumps(env, indent=2))
stage('setup_repo', t0)

def run_logged(cmd, log):
    """Run, print and save the output; return the wall time in seconds."""
    print('$', ' '.join(map(str, cmd)))
    t0 = time.time()
    with open(log, 'w') as fh:
        p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO_DIR)
        for line in p.stdout:
            print(line, end=''); fh.write(line); fh.flush()
        rc = p.wait()
    if rc: raise SystemExit(f'exit {rc}: see {log}')
    return round(time.time() - t0, 1)

common = ['--model', MODEL, '--revision', REVISION, '--formats', FORMATS, '--ngl', '99',
          '--llama-bin', LLAMA_BIN, '--work', WORK]

## Command 1: the 16-item sample (as #13, #18, #19, #23)

Download, the bf16 convert, six quantizations, then each file served with `llama-server` and scored on the script's
seeded 16-item sample of the stand-in set (standin-v0: 100 MMLU + 100 GSM8K test items, seed 0). The model name has no
"instruct" or "chat", so the script uses few-shot completion prompts (Sarvam-1 is a base model).

In [ ]:
t0 = time.time()
run_logged([sys.executable, QE, *common, '--limit', '16',
            '--runs-csv', OUT / 'runs_limit16.csv', '--scores-out', OUT / 'scores_limit16.json'],
           OUT / 'command1_limit16.log')
stage('command1_16_items', t0)

## Command 2: all 200 items

The same files (the script reuses each GGUF and its recorded quantize time), scored on the whole stand-in set: one
score has a standard error of about 3.4 points on 200 items, against about 12 on 16.

In [ ]:
if RUN_FULL:
    t0 = time.time()
    run_logged([sys.executable, QE, *common,
                '--runs-csv', OUT / 'runs_full.csv', '--scores-out', OUT / 'scores_full.json'],
               OUT / 'command2_full.log')
    stage('command2_200_items', t0)

## KL drift from bf16: English against Kannada

`quality/kl_eval.py` unchanged: `llama-perplexity --kl-divergence` on 64 chunks of 512 tokens (16,384 scored tokens),
bf16 as the reference. English = WikiText-2 test (the script's own text). Kannada = the first whole documents of
Sangraha `verified/kan` (pinned revision, CC-BY-4.0) up to 400,000 characters. The script reads
`<work>/<text>.test.txt` when it exists, so the Kannada text is written there and registered under the name
`sangraha-kn`; the script itself is not edited. Only the text's hash and size leave this machine.

In [ ]:
if RUN_KL:
    t0 = time.time()
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow', 'huggingface_hub'])
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem
    kn_txt = WORK / 'sangraha-kn.test.txt'
    if not kn_txt.exists():
        pf = pq.ParquetFile(HfFileSystem().open(
            f'datasets/ai4bharat/sangraha@{SANGRAHA_REV}/verified/kan/data-0.parquet', 'rb'))
        print('columns', pf.schema_arrow.names, '| rows', pf.metadata.num_rows)
        docs, n = [], 0
        for batch in pf.iter_batches(batch_size=256, columns=['text']):   # one 958 MB row group: read in batches
            for doc in batch.column('text').to_pylist():
                if doc and doc.strip():
                    docs.append(doc.strip()); n += len(docs[-1])
            if n >= KN_CHARS: break
        kn_txt.write_text('\n\n'.join(docs) + '\n', encoding='utf-8')
        kn_meta = dict(source=f'ai4bharat/sangraha@{SANGRAHA_REV} verified/kan/data-0.parquet, first docs in file order',
                       docs=len(docs), chars=len(kn_txt.read_text(encoding='utf-8')),
                       sha256=hashlib.sha256(kn_txt.read_bytes()).hexdigest())
        (OUT / 'kn_text.json').write_text(json.dumps(kn_meta, indent=2))
        print(kn_meta)
    stage('kl_text_prep', t0)

    wrapper = ROOT / 'kl_wrapper.py'   # registers the local Kannada text under a name, then runs kl_eval.main()
    wrapper.write_text(
        'import importlib.util, sys\n'
        f'spec = importlib.util.spec_from_file_location("kl_eval", r"{KL}")\n'
        'kl = importlib.util.module_from_spec(spec); spec.loader.exec_module(kl)\n'
        'kl.TEXTS["sangraha-kn"] = "<work>/sangraha-kn.test.txt, written by the notebook"\n'
        'kl.main()\n')
    for text in ('wikitext2', 'sangraha-kn'):
        t0 = time.time()
        run_logged(
            [sys.executable, wrapper, '--model', MODEL, '--revision', REVISION, '--text', text,
             '--chunks', KL_CHUNKS, '--work', WORK, '--llama-bin', LLAMA_BIN, '--out', OUT / f'kl_{text}.csv',
             '--notes', 'rotation subramanyanaik'],
            OUT / f'kl_{text}.log')
        stage(f'kl_{text}', t0)
        for f in WORK.glob(f'*-{text}-*.kld'): f.unlink()   # the 2 GB reference log-probabilities

## Summary: the tables for `runs/rotation_subramanyanaik.md`, the GPU use and the cost

In [ ]:
import csv
sampler.terminate(); gpu_log.close()
session_h = (time.time() - T_SESSION) / 3600
rows = [l.split(', ') for l in open(OUT / 'gpu_log.csv') if l.count(',') == 2]
mem = [float(r[1]) for r in rows]; util = [float(r[2]) for r in rows]
gpu_use = dict(peak_mem_gb=round(max(mem) / 1024, 2) if mem else None,
               mean_util_pct=round(sum(util) / len(util), 1) if util else None, samples=len(rows))
cost = dict(session_h=round(session_h, 2), units_per_hour=COLAB_UNITS_PER_HOUR,
            units=round(session_h * COLAB_UNITS_PER_HOUR, 2) if COLAB_UNITS_PER_HOUR else 0)
STAGES.update(session_h=cost['session_h']); (OUT / 'timings.json').write_text(json.dumps(STAGES, indent=2))
(OUT / 'gpu_cost.json').write_text(json.dumps(dict(gpu_use, **cost), indent=2))

def table(path):
    s = json.load(open(path))
    print(f"\n{s['eval_set']}, {s['n_items']} items, prompt {s['prompt_mode']}, context {s['context']} | {s['runtime']} | {s['machine']}")
    print('| Format | Size | Bits per weight | Score | By kind | Quantize | Eval | Tokens per second | Router | Output | sha256 |')
    print('| --- | ---: | ---: | ---: | --- | ---: | ---: | ---: | --- | --- | --- |')
    for f in s['formats']:
        kinds = ', '.join(f'{k} {v}' for k, v in f['score_by_kind'].items())
        print(f"| {f['format']} | {f['size_gb']} GB | {f['bpw']} | {f['score']} | {kinds} | {f['quant_time_s']} s | "
              f"{f['eval_time_s']} s | {f['gen_tok_s']} | {f['router_type']} | {f['output_type']} | `{f['sha256']}` |")

for p in ('scores_limit16.json', 'scores_full.json'):
    if (OUT / p).exists(): table(OUT / p)
for text in ('wikitext2', 'sangraha-kn'):
    p = OUT / f'kl_{text}.csv'
    if not p.exists(): continue
    print(f'\nKL from bf16 on {text} ({KL_CHUNKS} x 512, {KL_CHUNKS * 256:,} scored tokens)')
    print('| Format | Size | KL mean | KL 99th pct | Same top-1 | PPL | PPL ratio | Pass time |')
    print('| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |')
    for r in csv.DictReader(open(p)):
        print(f"| {r['format']} | {r['size_gb']} GB | {r['kld_mean'] or '-'} | {r['kld_99'] or '-'} | {r['same_top_p']}% | "
              f"{r['ppl']} | {r['ppl_ratio']} ± {r['ppl_ratio_unc'] or 0} | {r['time_s'] or '-'} s |")
print('\nstages (s):', json.dumps(STAGES, indent=1))
print('GPU:', gpu_use, '| cost:', cost)

In [ ]:
# Zip the small outputs (scores, CSVs, logs, timings; no GGUF, no model text) and download
import zipfile
z = ROOT / 'g12_rotation_sarvam1_out.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUT.rglob('*'):
        if f.is_file(): zf.write(f, f'out/{f.relative_to(OUT)}')
    for f in sorted({*WORK.glob('*.log'), *(WORK / 'kl-logs').glob('*.log')}):   # quantize-<fmt>.log, llama-server.log
        zf.write(f, f'work/{f.relative_to(WORK)}')
    for f in WORK.glob('*-scores.json'):
        zf.write(f, f'work/{f.name}')
print(z, round(z.stat().st_size / 1e6, 2), 'MB')
try:
    from google.colab import files
    files.download(str(z))
except ImportError:
    pass